# Cloud Gaming Android Builder

Backup/interactive builder for `atifbukhari/Cloud-Gaming-Client`. GitHub Actions remains the authoritative CI path.

Default branch: `demo-client-v1`. Produces the non-root debug APK used by the demo client.


In [ ]:
REPO = "https://github.com/atifbukhari/Cloud-Gaming-Client.git"
BRANCH = "demo-client-v1"
SDK_ROOT = "/content/android-sdk"
NDK_VERSION = "29.0.14206865"
print(REPO, BRANCH, NDK_VERSION)


In [ ]:
%%bash
set -euxo pipefail
apt-get update -qq
apt-get install -y -qq unzip wget git openjdk-17-jdk-headless
rm -rf /content/android-sdk
mkdir -p /content/android-sdk/cmdline-tools
wget -q https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip -O /content/cmdline-tools.zip
unzip -q /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest
export ANDROID_SDK_ROOT=/content/android-sdk
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses >/dev/null || true
/content/android-sdk/cmdline-tools/latest/bin/sdkmanager \
  "platform-tools" \
  "platforms;android-36" \
  "build-tools;36.0.0" \
  "ndk;29.0.14206865"


In [ ]:
%%bash
set -euxo pipefail
rm -rf /content/Cloud-Gaming-Client
git clone --branch demo-client-v1 --single-branch --recurse-submodules https://github.com/atifbukhari/Cloud-Gaming-Client.git /content/Cloud-Gaming-Client
cd /content/Cloud-Gaming-Client
git submodule update --init --recursive
printf 'sdk.dir=/content/android-sdk\nndk.dir=/content/android-sdk/ndk/29.0.14206865\n' > local.properties
chmod +x gradlew
export ANDROID_SDK_ROOT=/content/android-sdk
export ANDROID_HOME=/content/android-sdk
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
./gradlew :app:assembleNonRootDebug --stacktrace


In [ ]:
from pathlib import Path
import hashlib, shutil
from google.colab import files

apks = list(Path('/content/Cloud-Gaming-Client/app/build/outputs/apk/nonRoot/debug').glob('*.apk'))
assert apks, 'No APK found'
src = apks[0]
dst = Path('/content/cloud-gaming-demo-apk.apk')
shutil.copy2(src, dst)
digest = hashlib.sha256(dst.read_bytes()).hexdigest()
print('APK:', dst)
print('Size:', dst.stat().st_size)
print('SHA-256:', digest)
files.download(str(dst))


## Optional Google Drive copy

If you want a persistent copy, mount Drive and copy `/content/cloud-gaming-demo-apk.apk` into a chosen Drive folder. Do not store signing secrets in this notebook.
